In [ ]:
# Colab Cell 1: Backend Setup and Execution

!pip install flask flask-cors youtube-transcript-api transformers torch pyngrok

from flask import Flask, request, jsonify
from flask_cors import CORS
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    VideoUnavailable,
    TranscriptsDisabled
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import os
from pyngrok import ngrok
import traceback

# Set your ngrok authtoken here
# IMPORTANT: Replace "YOUR_NGROK_AUTH_TOKEN" with your actual token
# (You've already provided it in the prompt, so it's included below.)
NGROK_AUTH_TOKEN = "32yDsiGkcPH073xqhYcBzjY8DIC_2aTxCwbZseaaQtwg2sPXh"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = Flask(__name__)
CORS(app) # Enable CORS for all routes

# -----------------------------
# Load BERT fake news detection model
# -----------------------------
MODEL_NAME = "mrm8488/bert-tiny-finetuned-fake-news-detection"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# -----------------------------
# Helper functions
# -----------------------------
def analyze_line(text):
    """Analyze a single line of transcript for fake news / real info"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    labels = ["REAL", "FAKE"]

    if labels[torch.argmax(probs)] == "FAKE":
        misinfo_label = "MISINFORMATION"
    else:
        misinfo_label = "REAL"

    return {"label": misinfo_label, "score": torch.max(probs).item()}


def extract_video_id(url):
    """Extract YouTube video ID from URL"""
    if "v=" in url:
        return url.split("v=")[1].split("&")[0]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1].split("?")[0]
    else:
        return None


# -----------------------------
# Routes
# -----------------------------
@app.route('/analyze', methods=['POST'])
def analyze_video():
    data = request.json
    video_url = data.get("url")

    if not video_url:
        return jsonify({"error": "No URL provided"}), 400

    video_id = extract_video_id(video_url)
    if not video_id:
        return jsonify({"error": "Invalid YouTube URL"}), 400

    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id)
    except NoTranscriptFound:
        print(f"ERROR: No transcript found for video ID: {video_id}")
        return jsonify({"error": "No transcript available for this video (or it's in an unsupported language)."}), 500
    except VideoUnavailable:
        print(f"ERROR: Video unavailable for ID: {video_id}")
        return jsonify({"error": "The YouTube video is unavailable or private."}), 500
    except TranscriptsDisabled:
        print(f"ERROR: Transcripts disabled for video ID: {video_id}")
        return jsonify({"error": "Transcripts are disabled for this YouTube video."}), 500
    except Exception as e:
    # This catches ParseError or any unexpected response
        print(f"UNEXPECTED ERROR fetching transcript for video ID {video_id}: {e}")
        traceback.print_exc() # This will print the full error traceback to your terminal
        return jsonify({"error": f"Transcript could not be parsed or video has no captions: {str(e)}"}), 500

    analyzed_transcript = []
    misconceptions = []
    for line in transcript:
        analysis = analyze_line(line["text"])
        line_data = {
            "timestamp": line["start"],
            "text": line["text"],
            "misinformation": analysis["label"],
            "score": analysis["score"]
        }
        analyzed_transcript.append(line_data)

        if analysis["label"] == "MISINFORMATION":
            misconceptions.append(line_data)

    return jsonify({
        "video_url": video_url,
        "transcript": analyzed_transcript,
        "misconceptions": misconceptions
    })


# -----------------------------
# Main: Run Flask app with ngrok
# -----------------------------
if __name__ == '__main__':
    port = 5000  # Flask runs on port 5000 by default in Colab

    # Kill any existing ngrok tunnels if the cell is re-run
    try:
        ngrok.kill()
    except Exception: # Changed to catch specific Exception
        pass # Ignore if no tunnels are running

    public_url = ngrok.connect(port).public_url
    print(f" * Public URL for your Flask app: {public_url}")
    # Store the public_url in an environment variable or a global variable
    # so the frontend can access it.
    os.environ['FLASK_PUBLIC_URL'] = public_url
    app.run(port=port, use_reloader=False, debug=True) # Added debug=True here

 * Public URL for your Flask app: https://ac2e8659d3f7.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
# Import necessary modules for displaying HTML in Colab
from IPython.display import HTML, display
import os

# Retrieve the public URL from the environment variable set by the backend
# This ensures the frontend connects to the correct Flask instance
flask_public_url = os.environ.get('FLASK_PUBLIC_URL', 'http://localhost:8000')
# Fallback to localhost if FLASK_PUBLIC_URL isn't set (e.g., if backend cell wasn't run first)

# The complete HTML, CSS, and JavaScript, with the apiBaseUrl dynamically set
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>YouTube Fact Checker</title>
    <style>
        /* Paste the entire content of styles.css here */
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}

        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            color: #333;
        }}

        .container {{
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
        }}

        .header {{
            text-align: center;
            margin-bottom: 40px;
            color: white;
        }}

        .header h1 {{
            font-size: 2.5rem;
            margin-bottom: 10px;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
        }}

        .header p {{
            font-size: 1.1rem;
            opacity: 0.9;
        }}

        .input-section {{
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
            margin-bottom: 30px;
        }}

        .url-input-container {{
            display: flex;
            gap: 15px;
            margin-bottom: 20px;
        }}

        #youtubeUrl {{
            flex: 1;
            padding: 15px;
            border: 2px solid #e1e5e9;
            border-radius: 8px;
            font-size: 16px;
            transition: border-color 0.3s ease;
        }}

        #youtubeUrl:focus {{
            outline: none;
            border-color: #667eea;
            box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1);
        }}

        .analyze-btn {{
            background: linear-gradient(135deg, #667eea, #764ba2);
            color: white;
            border: none;
            padding: 15px 30px;
            border-radius: 8px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: transform 0.2s ease, box-shadow 0.2s ease;
        }}

        .analyze-btn:hover:not(:disabled) {{
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4);
        }}

        .analyze-btn:disabled {{
            opacity: 0.6;
            cursor: not-allowed;
        }}

        .loading {{
            display: none;
            text-align: center;
            margin: 20px 0;
        }}

        .spinner {{
            border: 3px solid #f3f3f3;
            border-top: 3px solid #667eea;
            border-radius: 50%;
            width: 30px;
            height: 30px;
            animation: spin 1s linear infinite;
            margin: 0 auto 10px;
        }}

        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}

        .results-section {{
            display: none;
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
            margin-bottom: 30px;
        }}

        .video-info {{
            display: flex;
            gap: 20px;
            margin-bottom: 30px;
            padding-bottom: 20px;
            border-bottom: 2px solid #f0f0f0;
        }}

        .video-thumbnail {{
            width: 200px;
            height: 112px;
            border-radius: 8px;
            object-fit: cover;
        }}

        .video-details h3 {{
            font-size: 1.3rem;
            margin-bottom: 10px;
            color: #333;
        }}

        .video-details p {{
            color: #666;
            margin-bottom: 5px;
        }}

        .tabs {{
            display: flex;
            margin-bottom: 20px;
            border-bottom: 2px solid #f0f0f0;
        }}

        .tab {{
            background: none;
            border: none;
            padding: 15px 25px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            color: #666;
            border-bottom: 3px solid transparent;
            transition: all 0.3s ease;
        }}

        .tab.active {{
            color: #667eea;
            border-bottom-color: #667eea;
        }}

        .tab-content {{
            display: none;
        }}

        .tab-content.active {{
            display: block;
        }}

        .transcript-section {{
            max-height: 400px;
            overflow-y: auto;
            border: 1px solid #e1e5e9;
            border-radius: 8px;
            padding: 20px;
        }}

        .transcript-line {{
            margin-bottom: 15px;
            padding: 10px;
            border-radius: 5px;
            position: relative;
        }}

        .transcript-line.has-misconception {{
            background-color: #fff5f5;
            border-left: 4px solid #f56565;
        }}

        .timestamp {{
            font-weight: 600;
            color: #667eea;
            font-size: 0.9rem;
        }}

        .text {{
            margin-top: 5px;
            line-height: 1.6;
        }}

        .misconception-badge {{
            background: #f56565;
            color: white;
            padding: 2px 8px;
            border-radius: 12px;
            font-size: 0.8rem;
            font-weight: 600;
            margin-left: 10px;
        }}

        .misconceptions-list {{
            display: grid;
            gap: 20px;
        }}

        .misconception-item {{
            background: #fff5f5;
            border: 1px solid #fed7d7;
            border-radius: 8px;
            padding: 20px;
        }}

        .misconception-header {{
            display: flex;
            align-items: center;
            margin-bottom: 15px;
        }}

        .severity-badge {{
            padding: 4px 12px;
            border-radius: 12px;
            font-size: 0.8rem;
            font-weight: 600;
            margin-right: 10px;
        }}

        .severity-high {{
            background: #f56565;
            color: white;
        }}

        .severity-medium {{
            background: #ed8936;
            color: white;
        }}

        .severity-low {{
            background: #ecc94b;
            color: #744210;
        }}

        .misconception-claim {{
            font-weight: 600;
            color: #333;
            margin-bottom: 10px;
        }}

        .fact-check {{
            background: #f0fff4;
            border: 1px solid #9ae6b4;
            border-radius: 5px;
            padding: 15px;
            margin-top: 10px;
        }}

        .fact-check-header {{
            font-weight: 600;
            color: #38a169;
            margin-bottom: 8px;
        }}

        .sources {{
            margin-top: 15px;
        }}

        .source-link {{
            color: #667eea;
            text-decoration: none;
            font-size: 0.9rem;
        }}

        .source-link:hover {{
            text-decoration: underline;
        }}

        .error-message {{
            background: #fed7d7;
            border: 1px solid #f56565;
            color: #c53030;
            padding: 15px;
            border-radius: 8px;
            margin: 20px 0;
            display: none;
        }}

        @media (max-width: 768px) {{
            .container {{
                padding: 15px;
            }}

            .url-input-container {{
                flex-direction: column;
            }}

            .video-info {{
                flex-direction: column;
            }}

            .video-thumbnail {{
                width: 100%;
                height: auto;
            }}

            .header h1 {{
                font-size: 2rem;
            }}

            .tabs {{
                flex-wrap: wrap;
            }}

            .tab {{
                padding: 10px 15px;
                font-size: 14px;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🔍 YouTube Fact Checker</h1>
            <p>Analyze YouTube videos for misconceptions and fact-check claims against official sources</p>
        </div>

        <div class="input-section">
            <div class="url-input-container">
                <input
                    type="url"
                    id="youtubeUrl"
                    placeholder="Paste YouTube URL here (e.g., https://www.youtube.com/watch?v=dQw4w9WgXcQ)"
                    required
                >
                <button class="analyze-btn" id="analyzeBtn">Analyze Video</button>
            </div>

            <div class="loading" id="loading">
                <div class="spinner"></div>
                <p>Analyzing video and checking facts...</p>
            </div>

            <div class="error-message" id="errorMessage"></div>
        </div>

        <div class="results-section" id="resultsSection">
            <div class="video-info" id="videoInfo">
                </div>

            <div class="tabs">
                <button class="tab active" data-tab="transcript">📝 Transcript</button>
                <button class="tab" data-tab="misconceptions">⚠️ Misconceptions</button>
                <button class="tab" data-tab="summary">📊 Summary</button>
            </div>

            <div class="tab-content active" id="transcript-content">
                <div class="transcript-section" id="transcriptContainer">
                    </div>
            </div>

            <div class="tab-content" id="misconceptions-content">
                <div class="misconceptions-list" id="misconceptionsList">
                    </div>
            </div>

            <div class="tab-content" id="summary-content">
                <div id="summaryContainer">
                    </div>
            </div>
        </div>
    </div>

    <script>
        class YouTubeFactChecker {{
            constructor() {{
                // IMPORTANT: This URL is dynamically inserted from the Python backend cell
                this.apiBaseUrl = '{flask_public_url}';
                this.initializeEventListeners();
            }}

            initializeEventListeners() {{
                const analyzeBtn = document.getElementById('analyzeBtn');
                const urlInput = document.getElementById('youtubeUrl');
                const tabs = document.querySelectorAll('.tab');

                analyzeBtn.addEventListener('click', () => this.analyzeVideo());
                urlInput.addEventListener('keypress', (e) => {{
                    if (e.key === 'Enter') this.analyzeVideo();
                }});

                tabs.forEach(tab => {{
                    tab.addEventListener('click', () => this.switchTab(tab.dataset.tab));
                }});
            }}

            extractVideoId(url) {{
                const regex = /(?:youtube\\.com\\/(?:[^\\/]+\\/.+\\/|(?:v|e(?:mbed)?)\\/|.*[?&]v=)|youtu\\.be\\/)([^"&?\\/\\s]{{11}})/;
                const match = url.match(regex);
                return match ? match[1] : null;
            }}

            showError(message) {{
                const errorDiv = document.getElementById('errorMessage');
                errorDiv.textContent = message;
                errorDiv.style.display = 'block';
            }}

            hideError() {{
                document.getElementById('errorMessage').style.display = 'none';
            }}

            showLoading() {{
                document.getElementById('loading').style.display = 'block';
                document.getElementById('analyzeBtn').disabled = true;
            }}

            hideLoading() {{
                document.getElementById('loading').style.display = 'none';
                document.getElementById('analyzeBtn').disabled = false;
            }}

            async analyzeVideo() {{
                const url = document.getElementById('youtubeUrl').value.trim();
                if (!url) {{
                    this.showError('Please enter a YouTube URL');
                    return;
                }}

                const videoId = this.extractVideoId(url);
                if (!videoId) {{
                    this.showError('Please enter a valid YouTube URL');
                    return;
                }}

                this.hideError();
                this.showLoading();

                try {{
                    const response = await fetch(`${{this.apiBaseUrl}}/analyze`, {{
                        method: 'POST',
                        headers: {{ 'Content-Type': 'application/json' }},
                        body: JSON.stringify({{ url }})
                    }});

                    if (!response.ok) {{
                        const errorData = await response.json();
                        throw new Error(errorData.error || `HTTP error! status: ${{response.status}}`);
                    }}

                    const data = await response.json();
                    this.displayResults(data);
                    document.getElementById('resultsSection').style.display = 'block';

                }} catch (error) {{
                    console.error('Analysis failed:', error);
                    this.showError(error.message || 'Analysis failed. Please try again later.');
                }} finally {{
                    this.hideLoading();
                }}
            }}

            displayResults(data) {{
                this.displayTranscript(data.transcript);
                this.displayMisconceptions(data.misconceptions);
                // The summary tab is a placeholder for future enhancement
                // this.displaySummary(data.summary);
            }}

            displayTranscript(transcript) {{
                const container = document.getElementById('transcriptContainer');
                container.innerHTML = transcript.map(line => `
                    <div class="transcript-line ${{line.misinformation === 'MISINFORMATION' ? 'has-misconception' : ''}}">
                        <div class="timestamp">${{this.formatTime(line.timestamp)}}></div>
                        <div class="text">
                            ${{line.text}}
                            ${{line.misinformation === 'MISINFORMATION' ? '<span class="misconception-badge">Potential Misconception</span>' : ''}}
                        </div>
                    </div>
                `).join('');
            }}

            displayMisconceptions(misconceptions) {{
                const container = document.getElementById('misconceptionsList');

                if (misconceptions.length === 0) {{
                    container.innerHTML = '<p style="text-align: center; color: #38a169; font-size: 1.1rem;">🎉 No misconceptions detected!</p>';
                    return;
                }}

                container.innerHTML = misconceptions.map(item => `
                    <div class="misconception-item">
                        <div class="misconception-header">
                            <span class="severity-badge severity-high">HIGH</span>
                            <span class="timestamp">At ${{this.formatTime(item.timestamp)}}</span>
                        </div>
                        <div class="misconception-claim">${{item.text}}</div>
                        <div class="fact-check">
                            <div class="fact-check-header">⚠️ Predicted as Misinformation</div>
                            <p>Confidence: ${{(item.score * 100).toFixed(2)}}%</p>
                        </div>
                    </div>
                `).join('');
            }}

            switchTab(tabName) {{
                document.querySelectorAll('.tab').forEach(tab => tab.classList.remove('active'));
                document.querySelectorAll('.tab-content').forEach(content => content.classList.remove('active'));

                document.querySelector(`[data-tab="${{tabName}}"]`).classList.add('active');
                document.getElementById(`${{tabName}}-content`).classList.add('active');
            }}

            formatTime(seconds) {{
                const min = Math.floor(seconds / 60);
                const sec = Math.floor(seconds % 60).toString().padStart(2, '0');
                return `${{min}}:${{sec}}`;
            }}
        }}

        document.addEventListener('DOMContentLoaded', () => {{
            new YouTubeFactChecker();
        }});
    </script>
</body>
</html>
"""

# Display the HTML in the Colab output
display(HTML(html_content))

In [ ]:
python3 -m pip install --upgrade youtube-transcript-api
